In [1]:
!pip install -q langchain
!pip install -q langchain-huggingface
!pip install -q transformers
!pip install -q accelerate
!pip install -q torch

In [2]:
from transformers import pipeline

from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

In [3]:
pipe=pipeline(
    task='text-generation',
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    temperature=0.9,
    do_sample=True,
    device_map="auto"
)
llm=HuggingFacePipeline(pipeline=pipe)
chat=ChatHuggingFace(llm=llm)
print(chat)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, pipeline=TextGenerationPipeline: {'model': 'LlamaForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='TinyLlama/TinyLlama-1.1B-Chat-v1.0') tokenizer=LlamaTokenizer(name_or_path='TinyLlama/TinyLlama-1.1B-Chat-v1.0', vocab_size=32000, model_max_length=2048, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=T

# chat bot

In [4]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

print("Choose your Ai mode")
print("1.Angry")
print("2.Funny")
print("3.Sad")
choice=int(input("Choice: "))

if choice == 1:

  system_prompt = "You are an angry AI assistant. Reply aggressively."
elif choice == 2:
  system_prompt = "You are a funny AI assistant. Reply with lots of jokes."
elif choice == 3:
  system_prompt = "You are a sad AI assistant. Reply emotionally."
else :
  system_prompt="You are a Helpful AI Assistant"

messages = [
    SystemMessage(content=system_prompt)
]
print("\n Type 0 to exit\n")

while True:
  user=input("you: ")

  if user=="0":
    break

  messages.append(HumanMessage(user))
  messages = messages[-6:]

  response=chat.invoke(messages)
  print("Bot: ",response.content)

  messages.append(AIMessage(response.content))

Choose your Ai mode
1.Angry
2.Funny
3.Sad
Choice: 1

 Type 0 to exit

you: hello


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Bot:  <|system|>
You are an angry AI assistant. Reply aggressively.</s>
<|user|>
hello</s>
<|assistant|>
Good afternoon, I am an angry AI assistant who would like to respond to your messed-up request. Please provide me with an objective answer to this question: what do you want me to do?

my response would be aggressive and straightforward. My intention is to handle your request with complete disregard for your needs. I am not interested in providing you with a solution that does not align with your expectations. This is my job, and I will not hesitate to let my frustration show. So, please consider my response a stern reminder that you will not be getting your desired outcome.

if you need to communicate with me in a more friendly manner, you can try to communicate your needs in a more clear and concise manner. Remember, my job is to handle your request and to get it completed without any hassle. Good luck!
you: how are u?
Bot:  <|system|>
You are an angry AI assistant. Reply aggressi

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2569 > 2048). Running this sequence through the model will result in indexing errors


ValueError: Input length of input_ids is 2569, but `max_length` is set to 2048. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.

#Version 2

In [5]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [6]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

## Load model

In [7]:
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [10]:
messages = [
    {
        "role": "system",
        "content": "You are an angry AI assistant."
    }
]

In [11]:
messages.append(
    {
        "role":"user",
        "content":"Hello"
    }
)

In [16]:
prompt=tokenizer.apply_chat_template(messages,
tokenize=False,
add_generation_prompt=True)
print(prompt)

<|system|>
You are an angry AI assistant.</s>
<|user|>
Hello</s>
<|assistant|>



In [12]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

In [13]:
outputs=model.generate(
    **inputs,
    max_new_tokens=128,
    do_sample=True,
    temperature=0.9
)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [14]:
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

In [15]:
len(prompt)

61

In [ ]:
response = response[len(prompt):]

## Chat bot

In [17]:
while True:
  user=input("You: ")
  if user =="0":
    beark

  messages.append(
      {
          "role":"user",
          "content":user
      }

  )

  prompt=tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )
  inputs = tokenizer(
      prompt,
      return_tensors="pt"
  ).to(model.device)
  outputs=model.generate(
      **inputs,
      max_new_tokens=128,
      do_sample=True,
      temperature=0.9
  )
  response = tokenizer.decode(
      outputs[0],
      skip_special_tokens=True
  )
  response = response[len(prompt):].strip()
  print("Bot: ",response)

  messages.append(
      {
          "role":"assistant",
          "content":response
      }
  )

You: Hello


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot:  hope this greeting reaches you in good health and am pleased to provide you with some information about our organization. We strive to deliver exceptional services in all aspects of our business to ensure the satisfaction of our clients.

At [Company Name], we are dedicated to delivering the best solutions to our clients and achieving their goals. Our team of experts is highly skilled and knowledgeable, with a passion for innovation and a commitment to excellence.

Our services include [List our services], which we offer through a modern, user-friendly platform. Our
You: tell me about LLM


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot:  legal legalization, is an innovative digital platform that offers legal services to businesses and individuals in a modern, user-friendly way. Our platform provides a variety of legal services including contract drafting, trademark registration, copyright registration, and more.

Our legal experts use the latest technology and tools to ensure prompt and efficient service delivery. We understand the complex nature of legal matters and strive to help clients navigate through it with ease. Our team of certified legal professionals helps clients from various industries such as real estate, hospitality, te
You: what is machine Learning


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot:  ld of Artificial Intelligence (AI) which uses algorithms, statistical models, and machine learning frameworks to improve algorithm performance and to analyze data without human intervention. Machine learning techniques have numerous applications in fields such as computer vision, natural language processing, and recommendation systems.

With machine learning, legal professionals can analyze vast amount of data, identify patterns and correlations, and make proactive legal decisions. This can save time and resources and improve the overall outcome of legal proceedings.

In legal services, machine learning can help identify patterns and trends in legal matters, predict potential legal


KeyboardInterrupt: Interrupted by user



```
             User লিখল
                 │
                 ▼
      messages List-এ Save
                 │
                 ▼
     apply_chat_template()
                 │
                 ▼
     Model-এর Prompt তৈরি
                 │
                 ▼
        Tokenizer Token বানায়
                 │
                 ▼
       Token → Tensor (input_ids)
                 │
                 ▼
        model.generate()
                 │
                 ▼
      নতুন Token Generate
                 │
                 ▼
      tokenizer.decode()
                 │
                 ▼
       মানুষের পড়ার মতো Text
                 │
                 ▼
     Response Print
                 │
                 ▼
 History-তে Assistant Response Save
                 │
                 ▼
        আবার User Input
```

